In [8]:
#@title 1. Configuración del Entorno y Carga de Datos
import pandas as pd
import plotly.express as px
import sys
import os

# Conectar el notebook con el módulo de utilidades (src/utils.py)
sys.path.append(os.path.abspath('..'))
from src.utils import set_global_style, cargar_datos

# Activar la identidad visual del portafolio
set_global_style()

# Cargar el dataset
df_fractura = cargar_datos("fractura_adjunto_iv.csv")

# Inspección de columnas para solucionar el KeyError
print("🔎 Columnas detectadas en el archivo CSV original:")
print(df_fractura.columns.tolist())

📥 Cargando datos desde '..\data\fractura_adjunto_iv.csv'...
✅ Dataset cargado. Dimensiones: (4890, 30)
💾 Memoria optimizada: de 4.18 MB a 3.80 MB
🔎 Columnas detectadas en el archivo CSV original:
['id_base_fractura_adjiv', 'idpozo', 'sigla', 'cuenca', 'areapermisoconcesion', 'yacimiento', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'longitud_rama_horizontal_m', 'cantidad_fracturas', 'tipo_terminacion', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'fecha_inicio_fractura', 'fecha_fin_fractura', 'fecha_data', 'anio_if', 'mes_if', 'anio_ff', 'mes_ff', 'anio_carga', 'mes_carga', 'empresa_informante', 'mes', 'anio']


In [9]:
#@title 2. Estandarización de Columnas y Definición de Alcance

# 1. Diccionario de estandarización
mapeo_columnas = {
    'cantidad_fracturas': 'etapas_fractura',
    'longitud_rama_horizontal_m': 'longitud_rama_horizontal',
    'agua_inyectada_m3': 'agua_bombeada_m3',
    'empresa_informante': 'operadora',
    'anio': 'anio_fractura'
}

# Renombramos las columnas en el dataframe
df_fractura = df_fractura.rename(columns=mapeo_columnas)
print("✅ Nombres de columnas estandarizados con éxito.")

# 2. Definición del Alcance (Filtro de Negocio)
# Según tu metodología: nos quedamos solo con pozos cuya rama horizontal sea >= 500 metros
# Usamos fillna(0) solo para la evaluación condicional y evitar errores con valores nulos
filtro_horizontales = df_fractura['longitud_rama_horizontal'].fillna(0) >= 500
df_horizontales = df_fractura[filtro_horizontales].copy()

# 3. Validación de resultados
print(f"📉 Registros totales iniciales: {len(df_fractura)}")
print(f"📈 Registros dentro del alcance analítico (rama >= 500m): {len(df_horizontales)}")
print("\nVista previa de los datos estandarizados:")
display(df_horizontales[['idpozo', 'operadora', 'longitud_rama_horizontal', 'etapas_fractura', 'tipo_reservorio']].head())

✅ Nombres de columnas estandarizados con éxito.
📉 Registros totales iniciales: 4890
📈 Registros dentro del alcance analítico (rama >= 500m): 2616

Vista previa de los datos estandarizados:


,idpozo,operadora,longitud_rama_horizontal,etapas_fractura,tipo_reservorio
2,159219,YPF S.A.,1437.300049,18,NO CONVENCIONAL
3,159220,YPF S.A.,1518.300049,19,NO CONVENCIONAL
4,159221,YPF S.A.,1482.300049,19,NO CONVENCIONAL
5,160426,YPF S.A.,1424.000000,18,NO CONVENCIONAL
6,160518,YPF S.A.,1504.000000,19,NO CONVENCIONAL


In [10]:
#@title 3. Consolidación, Ingeniería de Variables y Exportación

print("⚙️ Consolidando cargas parciales por pozo...")
# Agrupamos por pozo: sumamos volumen total y tomamos el máximo de las variables de diseño
cols_suma = ['arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_bombeada_m3']
cols_max = ['longitud_rama_horizontal', 'etapas_fractura', 'anio_fractura']

# Preparamos el diccionario de agregación validando de forma dinámica las columnas
agrupaciones = {col: 'sum' for col in cols_suma if col in df_horizontales.columns}
agrupaciones.update({col: 'max' for col in cols_max if col in df_horizontales.columns})

# Para mantener la operadora tras agrupar por pozo, tomamos su primer valor
if 'operadora' in df_horizontales.columns:
    agrupaciones['operadora'] = 'first'

df_consolidado = df_horizontales.groupby('idpozo', as_index=False).agg(agrupaciones)
print(f"✅ Pozos únicos tras consolidación: {len(df_consolidado)}")

print("🧪 Generando variables de intensidad operativa (Feature Engineering)...")
# Evitamos divisiones por cero descartando valores nulos o anómalos en los denominadores
df_consolidado = df_consolidado[df_consolidado['etapas_fractura'] > 0]
df_consolidado = df_consolidado[df_consolidado['longitud_rama_horizontal'] > 0]

# Suma total de arena (nacional + importada) manejando los valores nulos como ceros
nacional = df_consolidado.get('arena_bombeada_nacional_tn', 0).fillna(0)
importada = df_consolidado.get('arena_bombeada_importada_tn', 0).fillna(0)
df_consolidado['arena_total_tn'] = nacional + importada

# Cálculos de intensidad operativa
df_consolidado['arena_por_metro'] = df_consolidado['arena_total_tn'] / df_consolidado['longitud_rama_horizontal']
df_consolidado['agua_por_metro'] = df_consolidado.get('agua_bombeada_m3', 0) / df_consolidado['longitud_rama_horizontal']
df_consolidado['espaciamiento_etapas'] = df_consolidado['longitud_rama_horizontal'] / df_consolidado['etapas_fractura']

print("📈 Renderizando gráficos exploratorios interactivos con Plotly...\n")

# 1. Boxplot
fig1 = px.box(
    df_consolidado, 
    x='longitud_rama_horizontal',  
    y='operadora',                 
    color='operadora',
    orientation='h',               
    title='Estrategia Operativa: Longitud de Rama por Empresa',
    labels={'longitud_rama_horizontal': 'Longitud de Rama (m)', 'operadora': ''},
    hover_data=['idpozo'],
    height=600                     
)
# Ordenamos de mayor a menor mediana de arriba hacia abajo
fig1.update_layout(showlegend=False, yaxis={'categoryorder':'median ascending'})
fig1.show()

# 2. Scatter: Evolución de Intensidad en el Tiempo
import numpy as np

# Creamos una copia ligera para añadir "jitter" (desplazamiento horizontal aleatorio)
# Esto evita el solapamiento extremo de puntos en un mismo año
df_plot = df_consolidado.copy()
df_plot['anio_jitter'] = df_plot['anio_fractura'] + np.random.uniform(-0.15, 0.15, len(df_plot))

fig2 = px.scatter(
    df_plot, x='anio_jitter', y='arena_por_metro', color='operadora',
    size='longitud_rama_horizontal', hover_data=['idpozo', 'anio_fractura', 'etapas_fractura'],
    title='Evolución Histórica: Densidad de Arena por Metro (Tendencia Principal)',
    labels={'anio_jitter': 'Año de Fractura', 'arena_por_metro': 'Arena (tn/m)'},
    opacity=0.6,
    height=600
)

# 1. Cortamos el eje Y en 12 para que los 2 outliers no aplasten la visualización
fig2.update_yaxes(range=[0, 12])
# 2. Forzamos que el eje X siga mostrando años limpios a pesar del jitter
fig2.update_xaxes(tickmode='linear', dtick=2, tick0=2008)

# Posicionamiento de leyenda
fig2.update_layout(
    legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center", title=""),
    margin=dict(b=100)
)
fig2.show()

# 3. Matriz de Correlación 
cols_corr = ['longitud_rama_horizontal', 'etapas_fractura', 'arena_total_tn', 
             'agua_bombeada_m3', 'arena_por_metro', 'espaciamiento_etapas']
cols_corr = [col for col in cols_corr if col in df_consolidado.columns]

fig3 = px.imshow(
    df_consolidado[cols_corr].corr().round(2), text_auto=True, aspect="auto", 
    color_continuous_scale='RdBu_r', title='Matriz de Correlación: Variables de Diseño',
    height=600
)
fig3.show()

# Exportación del dataset procesado
import os
dir_procesados = os.path.join("..", "data", "processed")
os.makedirs(dir_procesados, exist_ok=True)
ruta_salida = os.path.join(dir_procesados, "pozos_analitico.csv")
df_consolidado.to_csv(ruta_salida, index=False)
print(f"💾 Dataset analítico limpio guardado en: {ruta_salida}")

⚙️ Consolidando cargas parciales por pozo...
✅ Pozos únicos tras consolidación: 2603
🧪 Generando variables de intensidad operativa (Feature Engineering)...
📈 Renderizando gráficos exploratorios interactivos con Plotly...



💾 Dataset analítico limpio guardado en: ..\data\processed\pozos_analitico.csv
